# Patrón Estructural: Adapter (Adaptador)

## Introducción
El patrón Adapter permite que dos interfaces incompatibles trabajen juntas. Convierte la interfaz de una clase en otra que el cliente espera.

## Objetivos
- Comprender cómo adaptar clases sin modificar su código.
- Identificar cuándo es útil el patrón Adapter.
- Comparar la solución con y sin el patrón.

## Ejemplo práctico
Supón que tienes una clase de impresora que espera un método `imprimir()`, pero tienes una clase de terceros con el método `print_document()`.

### Sin patrón Adapter (forma errónea)

In [1]:
class TerceroPrinter:
    def print_document(self, doc):
        print(f'Imprimiendo: {doc}')

def imprimir_documento(printer, doc):
    printer.imprimir(doc)

printer = TerceroPrinter()
# imprimir_documento(printer, 'Hola')  # Error: no tiene 'imprimir'

### Con patrón Adapter (forma correcta)

In [2]:
class PrinterAdapter:
    def __init__(self, adaptee):
        self.adaptee = adaptee
    def imprimir(self, doc):
        self.adaptee.print_document(doc)

printer = TerceroPrinter()
adapted = PrinterAdapter(printer)
imprimir_documento(adapted, 'Hola')

Imprimiendo: Hola


## UML del patrón Adapter
```plantuml
@startuml
class TerceroPrinter {
    + print_document(doc)
}
class PrinterAdapter {
    - adaptee: TerceroPrinter
    + imprimir(doc)
}
TerceroPrinter <.. PrinterAdapter : usa
@enduml
```

## Otro ejemplo de la vida real: Integración de una pasarela de pago de terceros
**Contexto:** tu sistema define una interfaz interna `procesar_pago(monto)` que toda tu app usa, con montos en formato decimal (ej. `49.99`). Pero al integrar Stripe, su SDK real no tiene ese método: expone `charge(amount_cents, currency)`, y además espera el monto en **centavos como entero** (así funciona la API real de Stripe). No puedes ni quieres modificar el SDK de Stripe, así que el Adapter traduce entre ambos mundos.

### Sin patrón (forma errónea)
El SDK de terceros no encaja con lo que tu código cliente espera.

In [3]:
class StripeSDK:
    def charge(self, amount_cents, currency='usd'):
        print(f'Cobrando {amount_cents} centavos ({currency}) vía Stripe')

def procesar_pago_pedido(procesador, monto):
    procesador.procesar_pago(monto)

stripe = StripeSDK()
# procesar_pago_pedido(stripe, 49.99)  # Error: StripeSDK no tiene 'procesar_pago', y tampoco maneja decimales

### Con patrón (forma correcta)
`StripeAdapter` implementa `procesar_pago(monto)` y por dentro traduce la llamada al formato que Stripe realmente espera (centavos enteros).

In [4]:
class StripeAdapter:
    def __init__(self, stripe_sdk):
        self._stripe = stripe_sdk
    def procesar_pago(self, monto):
        self._stripe.charge(amount_cents=int(monto * 100), currency='usd')

stripe = StripeSDK()
adaptador = StripeAdapter(stripe)
procesar_pago_pedido(adaptador, 49.99)

Cobrando 4999 centavos (usd) vía Stripe


### UML del ejemplo de pasarela de pago
```plantuml
@startuml
class StripeSDK {
    + charge(amount_cents, currency)
}
class StripeAdapter {
    - _stripe: StripeSDK
    + procesar_pago(monto)
}
StripeSDK <.. StripeAdapter : usa
@enduml
```

### ¿Dónde más se usa Adapter?
- **Integración de SDKs de terceros:** pasarelas de pago (Stripe, PayPal), servicios de email transaccional o mensajería, cada uno con su propia interfaz que hay que encajar en la tuya.
- **Migración incremental de sistemas legacy:** adaptar una API antigua (SOAP, XML) a la interfaz REST/JSON que espera el código nuevo, sin reescribir el sistema legacy de una vez.
- **Drivers de bases de datos:** una capa de acceso a datos que adapta distintos clientes (psycopg2, pymongo) a una interfaz común `Repositorio`.
- **Adaptar formatos de archivo:** convertir la interfaz de una librería que lee `.xlsx` a la interfaz genérica `LectorArchivo` que usa el resto del sistema.
- **Testing:** adaptar un objeto real complejo a una interfaz simple para poder sustituirlo fácilmente por un doble de prueba (fake/stub).

**Ejercicio de reflexión:** ¿qué pasaría si mañana cambias de Stripe a PayPal? ¿Cuántas clases de tu sistema tendrías que tocar si ya usas `StripeAdapter`, comparado con si el código cliente llamara directamente a `stripe.charge(...)` en 20 lugares distintos?

## Actividad
Crea tu propio Adapter para adaptar una clase de pagos de terceros a una interfaz de pagos esperada por tu sistema.